## Cell 1: Setup
**What this demonstrates**: Environment initialisation for Speculative RAG — two Anthropic models (Haiku for fast speculative drafting, Sonnet for careful verification), LangChain + Chroma for retrieval, and separate timing instrumentation to measure the fast vs slow path.
**Expected output**: Setup confirmation with model names, alignment threshold, and timing mode.

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb anthropic openai python-dotenv

import os
import re
import json
import time
import pathlib

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

DRAFT_MODEL         = 'claude-haiku-4-5-20251001'  # fast, cheap — speculative drafter
VERIFY_MODEL        = 'claude-sonnet-4-6'           # careful — verifier and regenerator
EMBED_MODEL         = 'text-embedding-3-small'
CHUNK_SIZE          = 400
CHUNK_OVERLAP       = 60
K_RETRIEVE          = 5
ALIGNMENT_THRESHOLD = 3   # score 1-5; >= threshold → fast path (speculation accepted)

client        = Anthropic()
lc_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Draft model         : {DRAFT_MODEL}')
print(f'  Verification model  : {VERIFY_MODEL}')
print(f'  Embed model         : {EMBED_MODEL}')
print(f'  Alignment threshold : {ALIGNMENT_THRESHOLD}/5 (>= → fast path)')
print(f'  K retrieve          : {K_RETRIEVE}')

## Cell 2: Data — Consumer Lending Policy
**What this demonstrates**: Loading a consumer lending policy as the knowledge base. The document contains both widely-known standards (minimum age = 18) that the model's training data likely covers, and institution-specific thresholds (exact charge-off timelines, specific hardship program caps) that the model cannot know without retrieval — creating a natural mix of fast-path and slow-path queries.
**Expected output**: Chunk count and contrast between queries the model can speculate on confidently vs. queries it cannot.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

policy_text = (BASE_DIR / 'fintech_policy.txt').read_text(encoding='utf-8')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', ' '],
)
policy_chunks = [d.page_content for d in splitter.create_documents([policy_text])]

print(f'Loaded: {len(policy_text):,} characters, {len(policy_chunks)} chunks')
print()
print('Query spectrum for Speculative RAG demonstration:')
print()
print('  COMMON (fast path expected):')
print('  "What is the minimum age for a personal loan?"')
print('  → Model likely knows: 18 (legal minimum to contract)')
print('  → Retrieval confirms Section 2.1 table: Age = 18 years or older')
print()
print('  NOVEL (slow path expected):')
print('  "What happens to collateral in a cross-default scenario?"')
print('  → Model uncertain: cross-default not a standard consumer lending term')
print('  → Retrieval finds Section 8 (default remediation + collateral rules)')
print('  → Verifier corrects speculation using actual policy language')
print()
print(f'Sample chunk:')
print('-' * 60)
print(policy_chunks[2][:350])

## Cell 3: Core — Speculative Draft, Retrieval, Verification, Routing
**What this demonstrates**: Four functions implement the full Speculative RAG pipeline. `speculative_answer` generates a fast draft with no documents. `retrieve_docs` fetches relevant context. `verify_alignment` asks the verifier to score the draft against the documents (1–5) and correct it if needed. `speculative_rag` wires these together: score >= ALIGNMENT_THRESHOLD → fast path (speculation returned); score < threshold → slow path (regenerate from docs).
**Expected output**: Index build confirmation and pipeline ready message.

In [ ]:
def speculative_answer(query: str) -> dict:
    """Generate a fast speculative draft from parametric knowledge — no retrieval.

    Uses DRAFT_MODEL (Haiku) for speed. The model is instructed to answer
    from training knowledge and flag uncertainty explicitly so the verifier
    can detect unreliable speculation.

    Args:
        query: User question.

    Returns:
        dict with keys: draft, latency_ms.
    """
    t0 = time.perf_counter()
    response = client.messages.create(
        model=DRAFT_MODEL,
        max_tokens=300,
        system=(
            'Answer concisely from your training knowledge. '
            'If you are uncertain about specific figures or institution-specific rules, '
            'say so explicitly — do not fabricate precise numbers.'
        ),
        messages=[{'role': 'user', 'content': query}],
    )
    return {
        'draft'      : response.content[0].text.strip(),
        'latency_ms' : (time.perf_counter() - t0) * 1000,
    }


def retrieve_docs(query: str, vectorstore: Chroma) -> dict:
    """Retrieve the top-k most relevant document chunks for a query.

    Args:
        query:       User question.
        vectorstore: Built Chroma index over the knowledge base.

    Returns:
        dict with keys: docs, latency_ms.
    """
    t0 = time.perf_counter()
    docs = vectorstore.similarity_search(query, k=K_RETRIEVE)
    return {
        'docs'       : docs,
        'latency_ms' : (time.perf_counter() - t0) * 1000,
    }


def verify_alignment(
    query: str,
    speculation: str,
    docs: list[Document],
) -> dict:
    """Score the speculative draft against retrieved documents and correct if needed.

    Asks VERIFY_MODEL to:
      - Score alignment 1-5 (1 = completely wrong, 5 = fully correct)
      - Identify specific errors or gaps
      - Return a corrected answer if score < ALIGNMENT_THRESHOLD

    The verifier responds in JSON for reliable parsing.

    Args:
        query:       Original user question.
        speculation: Speculative draft from speculative_answer.
        docs:        Retrieved documents from retrieve_docs.

    Returns:
        dict with keys: score, aligned, issues, answer, latency_ms.
    """
    context = '\n\n'.join(f'[Doc {i+1}]\n{d.page_content}' for i, d in enumerate(docs))

    prompt = (
        f'Query: {query}\n\n'
        f'Speculative answer (generated without documents):\n{speculation}\n\n'
        f'Retrieved documents:\n{context}\n\n'
        'Assess whether the speculative answer is accurate based on the documents.\n'
        'Respond with a JSON object — no other text — with these exact fields:\n'
        '{\n'
        '  "score": <integer 1-5>,\n'
        '  "issues": "<specific errors or gaps; empty string if none>",\n'
        '  "answer": "<confirmed answer if score>=3; corrected answer if score<3>"\n'
        '}'
    )

    t0 = time.perf_counter()
    response = client.messages.create(
        model=VERIFY_MODEL,
        max_tokens=500,
        messages=[{'role': 'user', 'content': prompt}],
    )
    latency_ms = (time.perf_counter() - t0) * 1000

    raw = response.content[0].text.strip()

    # Extract JSON — model may add surrounding text
    match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            score = int(parsed.get('score', 2))
            return {
                'score'      : score,
                'aligned'    : score >= ALIGNMENT_THRESHOLD,
                'issues'     : parsed.get('issues', ''),
                'answer'     : parsed.get('answer', speculation),
                'latency_ms' : latency_ms,
            }
        except (json.JSONDecodeError, ValueError):
            pass

    # Fallback: treat as not aligned, use raw response as answer
    return {
        'score'      : 2,
        'aligned'    : False,
        'issues'     : 'Could not parse verification response',
        'answer'     : raw,
        'latency_ms' : latency_ms,
    }


def speculative_rag(query: str, vectorstore: Chroma) -> dict:
    """Full Speculative RAG pipeline: draft → retrieve → verify → route.

    Fast path (score >= ALIGNMENT_THRESHOLD):
      Draft was accurate — the verified answer is returned. Latency is
      dominated by max(draft_time, retrieval_time) rather than their sum.

    Slow path (score < ALIGNMENT_THRESHOLD):
      Draft was inaccurate — the verifier's corrected answer (grounded in
      retrieved documents) is returned. Total latency is higher but
      accuracy is maintained.

    Args:
        query:       User question.
        vectorstore: Chroma index over the knowledge base.

    Returns:
        dict with keys: query, path, draft_result, retrieval_result,
        verification_result, answer, total_latency_ms.
    """
    t_pipeline = time.perf_counter()

    # Step 1: speculative draft — no documents, fast
    draft_result = speculative_answer(query)

    # Step 2: retrieve — runs after draft in this sequential implementation;
    # in production this would overlap with the draft call
    retrieval_result = retrieve_docs(query, vectorstore)

    # Step 3: verify draft against retrieved docs
    verification_result = verify_alignment(
        query,
        draft_result['draft'],
        retrieval_result['docs'],
    )

    path = 'fast' if verification_result['aligned'] else 'slow'

    return {
        'query'               : query,
        'path'                : path,
        'draft_result'        : draft_result,
        'retrieval_result'    : retrieval_result,
        'verification_result' : verification_result,
        'answer'              : verification_result['answer'],
        'total_latency_ms'    : (time.perf_counter() - t_pipeline) * 1000,
    }


# ── Build Chroma index from the policy document ───────────────────────────────

print('Building knowledge base index...', end=' ', flush=True)
policy_docs = [
    Document(page_content=chunk) for chunk in policy_chunks
]
vectorstore = Chroma.from_documents(
    policy_docs, lc_embeddings, collection_name='speculative_rag_policy'
)
print('done')
print(f'  {len(policy_chunks)} chunks indexed')
print()
print('Pipeline ready: speculative_rag(query, vectorstore)')
print(f'  Fast path: alignment score >= {ALIGNMENT_THRESHOLD} → speculation accepted')
print(f'  Slow path: alignment score < {ALIGNMENT_THRESHOLD} → regenerated from docs')

## Cell 4: Run — Common Query (Fast Path) vs Novel Query (Slow Path)
**What this demonstrates**: Two queries that exercise opposite routing paths. The common query (minimum age) uses widely-known legal standards the model's training data covers — the draft should be correct and the fast path taken. The novel query (cross-default collateral) involves institution-specific or uncertain policy details — the draft is likely incomplete and the slow path produces a grounded answer.
**Expected output**: Per query — path taken, alignment score, draft, and final answer.

In [ ]:
QUERIES = [
    (
        'What is the minimum age for a personal loan?',
        'common — model prior reliable (legal minimum to contract = 18)',
    ),
    (
        'What happens to collateral in a cross-default scenario?',
        'novel — cross-default not standard consumer lending; model uncertain',
    ),
]

all_results = []

for query, description in QUERIES:
    print(f'Query       : {query}')
    print(f'Description : {description}')
    print()

    result = speculative_rag(query, vectorstore)
    all_results.append(result)

    vr = result['verification_result']
    path_label = 'FAST PATH ✓' if result['path'] == 'fast' else 'SLOW PATH (corrected)'

    print(f'Path            : {path_label}')
    print(f'Alignment score : {vr["score"]}/5  (threshold = {ALIGNMENT_THRESHOLD})')
    if vr['issues']:
        print(f'Issues found    : {vr["issues"][:200]}')
    print()
    print('Speculative draft (no documents):')
    print('-' * 55)
    print(result['draft_result']['draft'])
    print()
    print('Final answer (verified):')
    print('=' * 55)
    print(result['answer'])
    print()
    print(f'Timing  draft={result["draft_result"]["latency_ms"]:.0f}ms  '
          f'retrieve={result["retrieval_result"]["latency_ms"]:.0f}ms  '
          f'verify={vr["latency_ms"]:.0f}ms  '
          f'total={result["total_latency_ms"]:.0f}ms')
    print()
    print('-' * 65)
    print()

## Cell 5: Inspect — Path Analysis, Timing Breakdown, Draft vs Final
**What this demonstrates**: The mechanics of each path in detail — timing breakdown per stage, alignment scores, a character-level diff between draft and final to show what the verifier changed, and a hypothetical latency comparison against standard RAG to quantify the fast-path benefit.
**Expected output**: Per-query path analysis with timing, draft-vs-final diff, and aggregate comparison table.

In [ ]:
# ── Per-query path inspection ──────────────────────────────────────────────────
for result in all_results:
    vr  = result['verification_result']
    dr  = result['draft_result']
    rr  = result['retrieval_result']

    print(f'QUERY: "{result["query"]}"')
    print(f'Path: {result["path"].upper()}')
    print('=' * 65)

    # Timing breakdown
    draft_ms   = dr['latency_ms']
    ret_ms     = rr['latency_ms']
    verify_ms  = vr['latency_ms']
    total_ms   = result['total_latency_ms']
    # Hypothetical parallel latency: draft and retrieval would overlap
    parallel_ms = max(draft_ms, ret_ms) + verify_ms

    print('Timing breakdown:')
    print(f'  Draft (Haiku, no docs)    : {draft_ms:6.0f} ms')
    print(f'  Retrieval (Chroma k={K_RETRIEVE})  : {ret_ms:6.0f} ms')
    print(f'  Verification (Sonnet)     : {verify_ms:6.0f} ms')
    print(f'  Total (sequential)        : {total_ms:6.0f} ms')
    print(f'  Total (parallel estimate) : {parallel_ms:6.0f} ms  ← production target')
    print()

    # Alignment score bar
    score = vr['score']
    bar   = chr(9608) * score + chr(9617) * (5 - score)
    print(f'Alignment score: {score}/5  [{bar}]  threshold={ALIGNMENT_THRESHOLD}')
    if vr['issues']:
        print(f'Issues: {vr["issues"][:250]}')
    print()

    # Draft vs final comparison
    draft_text = dr['draft']
    final_text = result['answer']
    changed    = draft_text.strip() != final_text.strip()

    print('Draft (pre-verification):')
    print(f'  {draft_text[:300].replace(chr(10), " ")}...' if len(draft_text) > 300 else f'  {draft_text.replace(chr(10), " ")}')
    print()
    print(f'Final answer ({"CHANGED by verifier" if changed else "CONFIRMED by verifier"}):')
    print(f'  {final_text[:300].replace(chr(10), " ")}...' if len(final_text) > 300 else f'  {final_text.replace(chr(10), " ")}')

    if changed:
        # Rough measure of how much the verifier changed things
        common_chars = sum(a == b for a, b in zip(draft_text, final_text))
        similarity   = common_chars / max(len(draft_text), len(final_text), 1)
        print(f'  Character similarity: {similarity:.0%}  ({"minor correction" if similarity > 0.7 else "major rewrite"})')

    print()
    print('-' * 65)
    print()

# ── Aggregate comparison table ─────────────────────────────────────────────────
print('PATH SUMMARY')
print('=' * 65)
print(f'{"Query":48s} {"Path":6s} {"Score":6s} {"Total ms":8s}')
print('-' * 65)
for result in all_results:
    q     = result['query'][:46]
    path  = result['path']
    score = result['verification_result']['score']
    ms    = result['total_latency_ms']
    print(f'{q:48s} {path:6s} {score:6d} {ms:8.0f}')

print()
print('Standard RAG estimate (draft latency replaced by wait-for-retrieval):')
for result in all_results:
    std_rag_ms = result['retrieval_result']['latency_ms'] + result['verification_result']['latency_ms']
    spec_ms    = max(result['draft_result']['latency_ms'],
                     result['retrieval_result']['latency_ms']) + result['verification_result']['latency_ms']
    saving_ms  = std_rag_ms - spec_ms
    q = result['query'][:50]
    print(f'  "{q}"')
    print(f'    Standard RAG est. {std_rag_ms:.0f}ms  |  Speculative (parallel) {spec_ms:.0f}ms  |  Saving {saving_ms:+.0f}ms')

## Cell 6: Fintech — Latency-Critical FAQ System
**What this demonstrates**: A simulated lending operations FAQ system handling multiple query types in sequence. Stable policy questions (eligibility criteria, product limits) should take the fast path; institution-specific or edge-case questions should take the slow path. The aggregate shows the latency benefit across a realistic query mix.
**Expected output**: Per-query path + timing, aggregate fast/slow split, and projected latency improvement for high-volume deployment.

In [ ]:
FAQ_QUERIES = [
    'What is the minimum credit score required to apply for a loan?',
    'What is the maximum loan amount for a personal loan?',
    'How many days past due before an account is charged off?',
    'What hardship accommodation programs are available for borrowers?',
    'What documentation is required for self-employed income verification?',
]

print('FINTECH: LENDING OPERATIONS FAQ — SPECULATIVE RAG')
print('Use case: operations team querying policy hundreds of times per day')
print('Goal: sub-second responses for stable policy questions')
print()

faq_results = []
for q in FAQ_QUERIES:
    print(f'  Running: "{q[:60]}"...', end=' ', flush=True)
    r = speculative_rag(q, vectorstore)
    faq_results.append(r)
    print(f'{r["path"]:4s} | score={r["verification_result"]["score"]}/5 | {r["total_latency_ms"]:.0f}ms')

print()
print('RESULTS')
print('=' * 65)
for r in faq_results:
    vr    = r['verification_result']
    path  = r['path']
    score = vr['score']
    ms    = r['total_latency_ms']
    icon  = '\u26a1' if path == 'fast' else '\u231b'
    q     = r['query'][:55]
    print(f'{icon} [{path:4s} | {score}/5] {q}')
    print(f'   Answer: {r["answer"][:180].replace(chr(10), " ")}...' if len(r['answer']) > 180 else f'   Answer: {r["answer"].replace(chr(10), " ")}')
    print()

# ── Aggregate stats ────────────────────────────────────────────────────────────
fast_results = [r for r in faq_results if r['path'] == 'fast']
slow_results = [r for r in faq_results if r['path'] == 'slow']

print('AGGREGATE STATS')
print('=' * 65)
print(f'  Total queries  : {len(faq_results)}')
print(f'  Fast path      : {len(fast_results)} ({100*len(fast_results)//len(faq_results)}%)')
print(f'  Slow path      : {len(slow_results)} ({100*len(slow_results)//len(faq_results)}%)')

if fast_results:
    avg_fast = sum(r['total_latency_ms'] for r in fast_results) / len(fast_results)
    print(f'  Avg fast-path latency  : {avg_fast:.0f} ms')
if slow_results:
    avg_slow = sum(r['total_latency_ms'] for r in slow_results) / len(slow_results)
    print(f'  Avg slow-path latency  : {avg_slow:.0f} ms')

all_latencies  = [r['total_latency_ms'] for r in faq_results]
# Standard RAG estimate: retrieval + verification (no parallel draft benefit)
std_rag_est    = [r['retrieval_result']['latency_ms'] + r['verification_result']['latency_ms']
                  for r in faq_results]

print()
print(f'  Avg actual latency     : {sum(all_latencies)/len(all_latencies):.0f} ms  (sequential)')
print(f'  Avg standard RAG est.  : {sum(std_rag_est)/len(std_rag_est):.0f} ms')
print()
print('Business case for high-volume FAQ (1,000 queries/day):')
fast_frac = len(fast_results) / len(faq_results)
print(f'  {fast_frac:.0%} of queries take the fast path.')
print(f'  For a system handling 1,000 queries/day, speculative RAG')
print(f'  eliminates retrieval latency from the critical path')
print(f'  for ~{int(fast_frac * 1000):,} queries per day.')
print()
print('Note: in production, draft generation and retrieval run in parallel.')
print('The sequential implementation above overstates total latency.')
print('Fast-path latency in production ≈ max(draft_ms, retrieval_ms) + verify_ms.')